In [ ]:
import pandas as pd
import numpy as np

In [ ]:
bm_raw = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/bm_raw_new.csv")
bss_pca = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/bss_pca_raw.csv")
pca_sellers = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/PCA for Sellers.csv")
classification = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/classification_mapping.xlsx")
Brand_mapping = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/Brand_mapping.xlsx")


In [ ]:
bss_redemption = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Redemption Reports/BSS Redemption.csv")

In [ ]:
free_credit = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Redemption Reports/Free-Credit-Data-Report_2026-01-14.csv")

In [ ]:
redemption_Consumption = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Redemption Reports/Redemption-Consumption-Data-Report_2026-01-14.csv")

In [ ]:
mtd_brand = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Redemption Reports/MTDOverBurnBrand.csv")

In [ ]:
mtd_seller = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Redemption Reports/MTDOverBurnSeller.csv")

# BM Raw

In [ ]:
bm_raw.columns

Index(['marketplace', 'campaign_id', 'advertiser_id', 'advertiser_name',
       'BUSINESS_ACCOUNT_ID', 'BUSINESS_ACCOUNT_NAME', 'seller_id',
       'adgroup_id', 'brand', 'commodity_name', 'analytic_vertical',
       'analytic_super_category', 'lifestyle_supercategory', 'Supercategory',
       'BU', 'HL_BU', 'Alpha_MP', 'BMP_Tag', 'Team', 'budget_type',
       'cost_model', 'pacing', 'campaign_type', 'campaign_name', 'status',
       'start_date', 'end_date', 'commodity_id', 'Channel', 'allocated_budget',
       'gross_budget', 'uniqueviewcount', 'actioncount', 'total_cost_x',
       'addtobasketcount', 'viewcount', 'cnt_lid', 'attr_units',
       'attr_revenue', 'overburn_share', 'Actual_spend', 'Budget_adgroup',
       'ROI', 'CTR', 'CVR'],
      dtype='object')

In [ ]:
bm_raw = bm_raw[bm_raw['total_cost_x']>5]

In [ ]:
# bss_pca.columns

In [ ]:
#  pca_sellers.columns

In [ ]:
classification.columns

Index(['Super Categories_Tag', 'BU_Tag', 'Wrong Tagging', 'New Tag', 'Type',
       'Adv. Name', 'Tag', 'Remarks', 'Super Categories_HL', 'BU_HL', 'BU_HL2',
       'HL Supercat'],
      dtype='object')

In [ ]:
Brand_mapping.columns

Index(['Incorrect Account Name', 'Correct Name', 'BU', 'Concatenate',
       'Advertiser Name', 'Tag', 'Ad Account ID', 'Ad Account Name',
       'Business Account ID', 'Business Account Name', 'Vertical',
       'Current SC', 'New SC'],
      dtype='object')

- New Alpha/MP

In [ ]:
incorrect_brands = Brand_mapping['Ad Account ID'].unique()

bm_raw['New Alpha/MP'] = np.where(
    bm_raw['advertiser_id'].isin(incorrect_brands) | (bm_raw['Team'] == 'IC'),
    'IC',
    bm_raw['Alpha_MP']
)


In [ ]:
bm_raw['New Alpha/MP'].value_counts()

,count
New Alpha/MP,
MP,452982
Alpha,42970
IC,2047


- FCFF Alpha/MP

In [ ]:
bm_raw['FCFF Alpha/MP'] = np.where(bm_raw['marketplace'] == "HYPERLOCAL","HL",bm_raw['Alpha_MP'])

In [ ]:
bm_raw['FCFF Alpha/MP'].value_counts()

,count
FCFF Alpha/MP,
MP,454825
Alpha,36818
HL,6356


- New Supercat

In [ ]:
brand_map = dict(zip(Brand_mapping['Vertical'], Brand_mapping['New SC']))
sc_map = dict(zip(classification['Wrong Tagging'], classification['New Tag']))

conditions = [
    bm_raw['marketplace'] == "Grocery",
    bm_raw['analytic_vertical'].isin(brand_map.keys()),
    bm_raw['analytic_super_category'].isin(brand_map.keys())
]

choices = [
    "Grocery",
    bm_raw['analytic_vertical'].map(brand_map),
    bm_raw['analytic_super_category'].map(brand_map)
]

fallback = bm_raw['analytic_super_category'].map(sc_map).fillna(bm_raw['analytic_super_category'])

bm_raw['New Supercat'] = np.select(conditions, choices, default=fallback)


In [ ]:
# print(bm_raw['New Supercat'].value_counts().to_string())


- Supercategory matching with FCFF's Super category

In [ ]:
bm_raw['SC match FCFF'] = bm_raw['New Supercat'].isin(classification['Super Categories_Tag'])

In [ ]:
# bm_raw['SC match FCFF'].value_counts()

- New BU

In [ ]:
sc_bu_map = dict(zip(classification['Super Categories_Tag'], classification['BU_Tag']))

bm_raw['New BU'] = bm_raw['New Supercat'].map(sc_bu_map)

In [ ]:
print(bm_raw['New BU'].isnull().sum())

289


- Spend

In [ ]:
bm_raw['Spends'] = bm_raw['total_cost_x']

In [ ]:
bm_raw['Spends'].sum()

np.float64(1159516729.84)

- Brand

In [ ]:
brand_map = dict(zip(Brand_mapping['Incorrect Account Name'], Brand_mapping['Correct Name']))

bm_raw['lookup_key'] = bm_raw['advertiser_name'].astype(str) + bm_raw['analytic_super_category'].astype(str)

cond_a = bm_raw['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bm_raw['analytic_super_category'].isin(Brand_mapping['BU'])

bm_raw['Brand'] = np.where(
    cond_a & cond_b,
    bm_raw['advertiser_name'].map(brand_map),
    bm_raw['brand']
)
bm_raw['Brand'] = bm_raw['Brand'].fillna(bm_raw['brand'])


# BSS PCA

In [ ]:
bss_pca.columns

Index(['Campaign ID', 'Marketplace', 'BU', 'Supercategory', 'HL_BU', 'store',
       'analytic_super_category', 'Brands', 'Store Name',
       'creative_template_id', 'creative_type', 'costmodel', 'Ad Account ID',
       'Ad Account Name', 'Business Account ID', 'Business Account Name',
       'Team', 'Alpha_MP', 'BMP_Tag', 'Channel', 'lifestyle_supercategory',
       'spend', 'views', 'clicks', 'ppv', 'click_direct_units',
       'click_indirect_units', 'click_direct_revenue',
       'click_indirect_revenue', 'Product', 'Average CPC', 'CTR', 'ROI'],
      dtype='object')

In [ ]:
bss_pca = bss_pca[bss_pca['spend']>5].copy()

- New Alpha/MP

In [ ]:
incorrect_brands = Brand_mapping['Advertiser Name'].unique()

bss_pca['New Alpha/MP'] = np.where(
    bss_pca['Business Account Name'].isin(incorrect_brands) | (bss_pca['Team'] == 'IC'),
    'IC',
    bss_pca['Alpha_MP']
)


- New Supercat

In [ ]:
class_map = classification.drop_duplicates('Wrong Tagging').set_index('Wrong Tagging')['New Tag'].to_dict()

conditions = [ bss_pca['BU'] == "Grocery", bss_pca['BU'] == "Mobile", bss_pca['Supercategory'] == "PetCare", bss_pca['Supercategory'].isin(class_map.keys()) ]
choices = [ "Grocery", "Mobile", "Pets", bss_pca['Supercategory'].map(class_map) ]

bss_pca['New Supercat'] = np.select(conditions, choices, default=bss_pca['Supercategory'])


In [ ]:
bss_pca['New Supercat'].isnull().sum()

np.int64(0)

- Supercategory matching with FCFF's Super category

In [ ]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

In [ ]:
bss_pca['SC match FCFF'].value_counts()

,count
SC match FCFF,
True,31973
False,468


- New BU

In [ ]:
sc_bu_map = classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()

bss_pca['New BU'] = bss_pca['New Supercat'].map(sc_bu_map)

- Brand

In [ ]:
brand_map = Brand_mapping.drop_duplicates('Incorrect Account Name').set_index('Incorrect Account Name')['Correct Name'].to_dict()

bss_pca['lookup_key'] = bss_pca['Ad Account Name'].astype(str) + bss_pca['Supercategory'].astype(str)

cond_a = bss_pca['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bss_pca['Supercategory'].isin(Brand_mapping['BU'])

bss_pca['Brand'] = np.where(
    cond_a & cond_b,
    bss_pca['Ad Account Name'].map(brand_map),
    bss_pca['Brands']
)
bss_pca['Brand'] = bss_pca['Brand'].fillna(bss_pca['Brands'])

- FCFF Alpha/MP

In [ ]:
bss_pca['FCFF Alpha/MP'] = np.where(bss_pca['Alpha_MP'] == "IC","Alpha",np.where(bss_pca['Alpha_MP'] =="3P","MP",bss_pca['Alpha_MP']))

In [ ]:
bss_pca['BUxBrand'] = bss_pca['New BU'].astype(str) + bss_pca['Brands'].astype(str)
lookup_map = bss_pca.drop_duplicates('BUxBrand').set_index('BUxBrand')['New Supercat'].to_dict()

bss_pca['Check'] = bss_pca['BUxBrand'].map(lookup_map)

In [ ]:
bss_pca[bss_pca['New BU'] == 0 ]['spend'].sum()

np.float64(0.0)

- check and redo the Supercat and BU Mapping

In [ ]:
bss_pca['New Supercat'] = np.where(bss_pca['SC match FCFF'] == False,bss_pca['Check'], bss_pca['New Supercat'])

In [ ]:
sc_bu_map = classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()

bss_pca['New BU'] = bss_pca['New Supercat'].map(sc_bu_map)

In [ ]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

# PCA for Sellers

In [ ]:
pca_sellers.columns

Index(['campaign_id', 'seller_id', 'BU', 'Supercategory', 'Store Name',
       'analytic_super_category', 'lifestyle_supercategory', 'brand',
       'KAM/NKAM', 'BMP_Tag', 'Channel', 'status', 'adspends', 'views',
       'clicks', 'revenue', 'units', 'CTR', 'CVR', 'ROI'],
      dtype='object')

In [ ]:
pca_seller = pca_sellers.copy()

- Alpha/MP

In [ ]:
pca_seller['Alpha/MP'] = "MP"

- Campaign Type

In [ ]:
pca_seller['Campaign Type'] = "PCA"

- New Supercatagory

In [ ]:
sc_map_cls = classification.drop_duplicates('Wrong Tagging').set_index('Wrong Tagging')['New Tag'].to_dict()

pca_seller['New Supercatagory'] = pca_seller['Supercategory'].map(sc_map_cls).fillna(pca_seller['Supercategory'])

- Supercategory matching with FCFF's Super category

In [ ]:
pca_seller['SC match FCFF'] = pca_seller['New Supercatagory'].isin(classification['Super Categories_Tag'])

In [ ]:
pca_seller['SC match FCFF'].value_counts()

,count
SC match FCFF,
True,298156
False,7707


In [ ]:
print(round(pca_seller[pca_seller['SC match FCFF'] == True]['adspends'].sum()))

9361205


- BU

In [ ]:
sc_bu_map =classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()
pca_seller['BU'] = pca_seller['New Supercatagory'].map(sc_bu_map)

In [ ]:
#  pca_seller['BU'].value_counts()

In [ ]:
brand_sc_map =pca_seller.drop_duplicates('brand').set_index('brand')['Supercategory'].to_dict()
pca_seller['Check'] = pca_seller['brand'].map(brand_sc_map)

- Check and redo Supercat and BU mapping

In [ ]:
pca_seller['New Supercatagory'] = np.where(
    (pca_seller['SC match FCFF'] == False) & (pca_seller['New Supercatagory'] != 'Not assigned'),
    pca_seller['Check'],
    pca_seller['New Supercatagory']
)


In [ ]:
sc_bu_map =classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()
pca_seller['BU'] = pca_seller['New Supercatagory'].map(sc_bu_map)

In [ ]:
pca_seller['SC match FCFF'] = pca_seller['New Supercatagory'].isin(classification['Super Categories_Tag'])

In [ ]:
print(round(pca_seller[pca_seller['SC match FCFF'] == True]['adspends'].sum()))

9457362


# MTD Overburn Brand and Sellers

In [ ]:
brand_mtd = mtd_brand[["CampaignId","BudgetType","CampaignType","Overburn"]].copy()

In [ ]:
seller_mtd = mtd_seller[["CampaignId","BudgetType","CampaignType","Overburn"]].copy()

In [ ]:
mtd_overburn = pd.concat([brand_mtd,seller_mtd],axis=0)


In [ ]:
print(round(mtd_overburn['Overburn'].sum()))

1746457


In [ ]:
mtd_overburn.columns

Index(['CampaignId', 'BudgetType', 'CampaignType', 'Overburn'], dtype='object')

# BSS Rredemption

In [ ]:
bss_redemption.columns

Index(['campaign_id', 'upi_id', 'billing_period_hash', 'payment_type',
       'account_id', 'campaign_spend', 'campaign_redeem_fc', 'campaign_redeem',
       'campaign_name', 'campaign_start_date', 'campaign_end_date',
       'campaign_budget', 'campaign_type', 'budget_type', 'market_place',
       'cost_model', 'brands', 'super_category_meta'],
      dtype='object')

- Filtered BSS Redemption Report

In [ ]:
bss_redem = bss_redemption[bss_redemption['billing_period_hash'].astype(str).str.startswith('2026-01')].copy()


In [ ]:
round(bss_redem['campaign_spend'].sum())

668919999

In [ ]:
round(bss_redem['campaign_redeem_fc'].sum())

85172020

In [ ]:
round(bss_redem['campaign_redeem'].sum())

583747979

# Seller Portal Data

- Redemption&Consumption and FreeCredit Seller Data

In [ ]:
free_credit.columns

Index(['campaign_id', 'seller_id', 'amount', 'date', 'merchant_id',
       'compartment_type'],
      dtype='object')

In [ ]:
redemption_Consumption.columns

Index(['campaign_id', 'seller_id', 'amount', 'date', 'merchant_id'], dtype='object')

In [ ]:
fc = free_credit.copy()
rc = redemption_Consumption.copy()

- Subtracted FC amount from RC amount to get Net FC

In [ ]:
rc_unique = rc.groupby(['campaign_id', 'seller_id'])['amount'].sum().reset_index()
fc_unique = fc.groupby(['campaign_id', 'seller_id'])['amount'].sum().reset_index()

df_final = rc_unique.merge(fc_unique, on=['campaign_id', 'seller_id'], how='outer', suffixes=('_rc', '_fc'))

df_final['amount_rc'] = df_final['amount_rc'].fillna(0)
df_final['amount_fc'] = df_final['amount_fc'].fillna(0)

df_final['Net_amount'] = df_final['amount_rc'] - df_final['amount_fc']

rc_final = df_final[['campaign_id', 'seller_id', 'amount_rc', 'amount_fc', 'Net_amount']].copy()


In [ ]:
round(rc_final['amount_rc'].sum())

413402776

In [ ]:
round(rc_final['amount_fc'].sum())

71398423

In [ ]:
round(rc_final['Net_amount'].sum())

342004353

# Search Allocation

1. BM RAW


In [ ]:
# bm_raw.columns

In [183]:
# pca_seller.columns

- BM Raw matched with FCFF Mapping

In [ ]:
bm_raw_df = bm_raw[bm_raw['SC match FCFF'] == True][[ 'campaign_id', 'seller_id','FCFF Alpha/MP','New BU',
                    'analytic_vertical','campaign_type', 'Brand', 'Spends',
                     'Actual_spend','New Supercat','Team', 'BMP_Tag','advertiser_id' ]]

In [178]:
bm_raw_df = bm_raw_df.rename(columns={'FCFF Alpha/MP':'Alpha/MP','Team':'KAM/NKAM'})

In [179]:
bm_raw_df.columns

Index(['campaign_id', 'seller_id', 'Alpha/MP', 'New BU', 'analytic_vertical',
       'campaign_type', 'Brand', 'Spends', 'Actual_spend', 'New Supercat',
       'KAM/NKAM', 'BMP_Tag', 'advertiser_id'],
      dtype='object')

- PCA Sellers matched with FCFF Mapping

In [ ]:
pca_seller_df = pca_seller[pca_seller['SC match FCFF'] == True] \
                          [['campaign_id', 'seller_id','Alpha/MP', 'BU','Campaign Type',
                          'brand','adspends', 'New Supercatagory', 'KAM/NKAM','BMP_Tag' ]]

In [172]:
pca_seller_df['Actual_spend'] = pca_seller_df['adspends']

In [173]:
pca_seller_df['analytic_vertical'] = np.nan

In [175]:
pca_seller_df['AdvertiserID'] = np.nan

In [176]:
pca_seller_df.columns

Index(['campaign_id', 'seller_id', 'Alpha/MP', 'BU', 'Campaign Type', 'brand',
       'adspends', 'New Supercatagory', 'KAM/NKAM', 'BMP_Tag', 'Actual_spend',
       'analytic_vertical', 'AdvertiserID'],
      dtype='object')

In [180]:
pca_seller_df = pca_seller_df.rename(columns={'campaign_id':'campaign_id', 'seller_id':'seller_id', 'Alpha/MP':'Alpha/MP', 'BU':'New BU', 'Campaign Type':'campaign_type', 'brand':'Brand',
       'adspends':'Spends', 'New Supercatagory':'New Supercat', 'KAM/NKAM':'KAM/NKAM', 'BMP_Tag':'BMP_Tag', 'Actual_spend':'Actual_spend',
       'analytic_vertical':'analytic_vertical', 'AdvertiserID':'advertiser_id'})

In [182]:
pca_seller_df.columns

Index(['campaign_id', 'seller_id', 'Alpha/MP', 'New BU', 'campaign_type',
       'Brand', 'Spends', 'New Supercat', 'KAM/NKAM', 'BMP_Tag',
       'Actual_spend', 'analytic_vertical', 'advertiser_id'],
      dtype='object')